## Carga de Datos

In [ ]:
import pandas as pd

df = pd.read_csv("route_cost_impact.csv")

df.head()



,month,conflict_phase,airline,iata_code,origin_city,destination_city,aircraft_type,original_distance_km,actual_distance_km,extra_distance_km,...,brent_crude_usd,jet_fuel_usd_barrel,total_fuel_cost_usd,extra_fuel_cost_usd,base_ticket_price_usd,fuel_surcharge_usd,total_ticket_price_usd,estimated_passengers,route_revenue_usd,fuel_pct_of_cost
0,2019-01,Pre-Pandemic Baseline,Emirates,EK,Dubai,London,Boeing 777-300ER,5500,5500.0,0.0,...,63.00,74.58,8949.6,0.0,361.93,77.12,439.05,231,101420.55,97.02
1,2019-02,Pre-Pandemic Baseline,Emirates,EK,Dubai,London,Boeing 777-300ER,5500,5500.0,0.0,...,67.61,81.72,9806.4,0.0,396.58,92.60,489.18,259,126697.62,97.27
2,2019-03,Pre-Pandemic Baseline,Emirates,EK,Dubai,London,Boeing 777-300ER,5500,5500.0,0.0,...,65.86,76.87,9224.4,0.0,373.05,81.93,454.98,220,100095.60,97.11
3,2019-04,Pre-Pandemic Baseline,Emirates,EK,Dubai,London,Boeing 777-300ER,5500,5500.0,0.0,...,64.79,73.34,8800.8,0.0,355.91,74.58,430.49,263,113218.87,96.97
4,2019-05,Pre-Pandemic Baseline,Emirates,EK,Dubai,London,Boeing 777-300ER,5500,5500.0,0.0,...,61.25,72.97,8756.4,0.0,354.12,73.83,427.95,283,121109.85,96.96


# Auditoría Inicial

In [ ]:
df.info()

#Ahora vamos a ver cuantos nulos hay por columna con la funcion .isnull()

print("Nulos por columna: ")
print(df.isnull().sum())

#Ahora vamos a ver si hay filas duplicadas con la funcion .duplicated()

print("Filas duplicadas")

print("Cantidad de filas duplicadas: ", df.duplicated().sum())

# Limpieza y Preprocesamiento

In [ ]:
df_clean = df.drop_duplicates()

In [ ]:
#Ahora rellenamos los nulos, que en este caso no tenemos, pero lo vamos a hacer igual

promedio_combustible = df_clean['jet_fuel_usd_barrel'].mean()
df_clean['jet_fuel_usd_barrel'] = df_clean['jet_fuel_usd_barrel'].fillna(promedio_combustible)

promedio_ticket = df_clean['total_ticket_price_usd'].mean()
df_clean['total_ticket_price_usd'] = df_clean['total_ticket_price_usd'].fillna(promedio_ticket)

#Tambien vamos a arreglar los formatos de texto de los meses a formato date

df_clean['month'] = pd.to_datetime(df_clean['month'])

print("Limpieza terminada con éxito")

¡Limpieza terminada con éxito!


## Detección de Valores Atípicos (Outliers)

In [ ]:
import pandas as pd
import numpy as np


df_clean = pd.read_csv("route_cost_impact_limpio.csv")
# Lista exacta de las columnas numéricas relevantes de tu archivo
columnas_a_revisar = [
    'actual_distance_km', 'extra_distance_km', 'fuel_consumption_bbl',
    'brent_crude_usd', 'jet_fuel_usd_barrel', 'total_fuel_cost_usd',
    'extra_fuel_cost_usd', 'total_ticket_price_usd', 'fuel_surcharge_usd',
    'estimated_passengers', 'route_revenue_usd', 'fuel_pct_of_cost'
]

print("REPORTE DE OUTLIERS (MÉTODO IQR)")
print("Detectando valores extremos en el dataset limpio...\n")

total_outliers_encontrados = 0

for col in columnas_a_revisar:
    # Calcular los cuartiles
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1

    # Calcular límites
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    # Filtrar outliers
    outliers = df_clean[(df_clean[col] < limite_inferior) | (df_clean[col] > limite_superior)]
    cantidad = len(outliers)

    if cantidad > 0:
        print(f" {col}: {cantidad} valores atípicos detectados.")
        total_outliers_encontrados += cantidad

print(f"\nTotal general de datos atípicos detectados: {total_outliers_encontrados}")
print("Nota técnica: No se eliminarán debido a que representan eventos extremos reales (ej. Pandemia/Conflictos).")

--- REPORTE DE OUTLIERS (MÉTODO IQR) ---
Detectando valores extremos en el dataset limpio...

🔹 actual_distance_km: 5 valores atípicos detectados.
🔹 extra_distance_km: 116 valores atípicos detectados.
🔹 fuel_consumption_bbl: 8 valores atípicos detectados.
🔹 brent_crude_usd: 216 valores atípicos detectados.
🔹 jet_fuel_usd_barrel: 180 valores atípicos detectados.
🔹 total_fuel_cost_usd: 54 valores atípicos detectados.
🔹 extra_fuel_cost_usd: 116 valores atípicos detectados.
🔹 total_ticket_price_usd: 62 valores atípicos detectados.
🔹 fuel_surcharge_usd: 135 valores atípicos detectados.
🔹 estimated_passengers: 68 valores atípicos detectados.
🔹 route_revenue_usd: 73 valores atípicos detectados.
🔹 fuel_pct_of_cost: 295 valores atípicos detectados.

Total general de datos atípicos detectados: 1328
Nota técnica: No se eliminarán debido a que representan eventos extremos reales (ej. Pandemia/Conflictos).


##  Validación y Exportación

In [ ]:
import pandas as pd
1
# 1. Cargar ambos archivos (asegúrate de que ambos estén subidos a Colab)
df_original = pd.read_csv('route_cost_impact.csv')
df_limpio = pd.read_csv('route_cost_impact_limpio.csv')

print("--- REPORTE DE CALIDAD: ANTES VS DESPUÉS ---\n")

# 2. Comparar el tamaño (para ver si se eliminaron filas duplicadas o nulas)
print("1. TAMAÑO DEL ARCHIVO (Filas, Columnas):")
print(f"Original: {df_original.shape}")
print(f"Limpio:   {df_limpio.shape}")

# 3. Comparar los datos nulos totales
print("\n2. CANTIDAD TOTAL DE CELDAS VACÍAS (NULOS):")
print(f"Original: {df_original.isnull().sum().sum()}")
print(f"Limpio:   {df_limpio.isnull().sum().sum()}")

# 4. Comparar duplicados
print("\n3. FILAS DUPLICADAS:")
print(f"Original: {df_original.duplicated().sum()}")
print(f"Limpio:   {df_limpio.duplicated().sum()}")

# 5. Comparar el formato de la columna de fechas
print("\n4. FORMATO DE LA COLUMNA 'month' (Primeros 3 registros):")
print(f"Original: {df_original['month'].head(3).tolist()}")
print(f"Limpio:   {df_limpio['month'].head(3).tolist()}")

--- REPORTE DE CALIDAD: ANTES VS DESPUÉS ---

1. TAMAÑO DEL ARCHIVO (Filas, Columnas):
Original: (3132, 23)
Limpio:   (3132, 23)

2. CANTIDAD TOTAL DE CELDAS VACÍAS (NULOS):
Original: 0
Limpio:   0

3. FILAS DUPLICADAS:
Original: 0
Limpio:   0

4. FORMATO DE LA COLUMNA 'month' (Primeros 3 registros):
Original: ['2019-01', '2019-02', '2019-03']
Limpio:   ['2019-01-01', '2019-02-01', '2019-03-01']


In [ ]:
#Ahora guardamos el archivo limpio
df_clean.to_csv('route_cost_impact_limpio.csv', index=False)
print("Archivo guardado.")

Archivo guardado.
